In [1]:
# Celda 1 — Cargar p5.js
from IPython.display import HTML
HTML("""
<script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.0/p5.min.js"></script>
<p style="color:#4caf50; font-weight:bold;">✅ p5.js cargado</p>
""")

In [2]:
# Celda 2 — Editor de Cuadros Interactivo
from IPython.display import HTML

HTML("""
<div style="
  font-family: sans-serif;
  background-color: #333333;
  padding: 20px;
  border-radius: 12px;
  display: inline-block;
  color: white;
">
  <p style="margin:0 0 12px 0; font-size:14px;">
    🖱️ <b>Clic</b> para iniciar el cuadro · <b>Mueve</b> el ratón · <b>Clic</b> para terminar · <b>Delete/Supr</b> para borrar
  </p>

  <!-- Botones de control -->
  <div style="margin-bottom:10px; display:flex; gap:10px; align-items:center; flex-wrap:wrap;">
    <button onclick="borrarTodo()"
      style="background:#e94560;color:white;border:none;padding:7px 16px;border-radius:8px;cursor:pointer;font-size:13px;font-weight:bold;">
      🗑️ Borrar todo
    </button>
    <label style="font-size:13px; display:flex; align-items:center; gap:6px; cursor:pointer;">
      Color:
      <input type="color" id="colorPicker" value="#0000ff"
        style="width:36px; height:28px; border:none; border-radius:6px; cursor:pointer;">
    </label>
    <label style="font-size:13px; display:flex; align-items:center; gap:6px;">
      Opacidad:
      <input type="range" id="opacitySlider" min="0.1" max="1" step="0.05" value="0.4"
        style="width:100px;"
        oninput="document.getElementById('opacVal').innerText=parseFloat(this.value).toFixed(2)">
      <span id="opacVal" style="font-size:12px;">0.40</span>
    </label>
    <span id="contadorSpan" style="font-size:12px; color:#aaa;">Cuadros: 0</span>
  </div>

  <!-- Canvas -->
  <div id="editor-canvas"
    style="background:#ffffff; border:2px solid #000; cursor:crosshair; display:inline-block;">
  </div>

  <!-- Info en tiempo real -->
  <div id="info-cuadros"
    style="margin-top:8px; font-size:12px; color:#cccccc; font-family:monospace; min-height:18px;">
  </div>
</div>

<script>
var editorSketch = function(p) {

  var W = 700, H = 500;

  // ── Estado del editor ──────────────────────────────────
  var cuadros        = [];
  var estadoDibujo   = 0;      // 0 = esperando, 1 = dibujando
  var inicioX, inicioY;
  var cuadroTemporal  = null;
  var cuadroHover     = null;

  // Leer color y opacidad del panel
  function getColor() {
    var hex  = document.getElementById('colorPicker').value;
    var opac = parseFloat(document.getElementById('opacitySlider').value);
    var r = parseInt(hex.slice(1,3),16);
    var g = parseInt(hex.slice(3,5),16);
    var b = parseInt(hex.slice(5,7),16);
    return { r:r, g:g, b:b, a:opac };
  }

  // Exponer funciones globales para los botones HTML
  window.borrarTodo = function() {
    cuadros = [];
    cuadroHover = null;
    document.getElementById('contadorSpan').innerText = 'Cuadros: 0';
  };

  // ── Setup ──────────────────────────────────────────────
  p.setup = function() {
    var cnv = p.createCanvas(W, H);
    cnv.parent('editor-canvas');
    p.noLoop();   // solo redibujamos cuando hay eventos
  };

  // ── Draw ───────────────────────────────────────────────
  p.draw = function() {
    p.background(255);

    // Cuadrícula sutil
    p.stroke(220);
    p.strokeWeight(1);
    for (var gx = 0; gx < W; gx += 40) p.line(gx, 0, gx, H);
    for (var gy = 0; gy < H; gy += 40) p.line(0, gy, W, gy);

    // Dibujar cuadros guardados
    for (var i = 0; i < cuadros.length; i++) {
      var c = cuadros[i];
      var esHover = (c === cuadroHover);

      p.fill(c.r, c.g, c.b, c.a * 255);
      p.stroke(esHover ? p.color(255,80,80) : p.color(0,0,180));
      p.strokeWeight(esHover ? 2.5 : 1.5);
      p.rect(c.x, c.y, c.ancho, c.alto);

      // Etiqueta de medidas al hacer hover
      if (esHover) {
        var lx = c.ancho < 0 ? c.x + c.ancho : c.x;
        var ly = c.alto  < 0 ? c.y + c.alto  : c.y;
        p.fill(0);
        p.noStroke();
        p.textSize(11);
        p.textAlign(p.LEFT);
        p.text(Math.abs(c.ancho) + ' × ' + Math.abs(c.alto) + ' px', lx + 4, ly + 14);
      }
    }

    // Cuadro temporal (mientras dibuja)
    if (cuadroTemporal) {
      var ct = cuadroTemporal;
      p.fill(ct.r, ct.g, ct.b, ct.a * 255);
      p.stroke(220, 50, 50);
      p.strokeWeight(2);
      p.rect(ct.x, ct.y, ct.ancho, ct.alto);

      // Medidas en tiempo real
      p.fill(0);
      p.noStroke();
      p.textSize(12);
      p.textAlign(p.LEFT);
      p.text('Ancho: ' + Math.abs(ct.ancho) + 'px   Alto: ' + Math.abs(ct.alto) + 'px',
             12, 20);
    }
  };

  // ── Clic izquierdo — iniciar / confirmar cuadro ────────
  p.mouseClicked = function() {
    if (p.mouseX < 0 || p.mouseX > W || p.mouseY < 0 || p.mouseY > H) return;

    if (estadoDibujo === 0) {
      // Primer clic: marca el inicio
      inicioX = p.mouseX;
      inicioY = p.mouseY;
      var col = getColor();
      cuadroTemporal = { x:inicioX, y:inicioY, ancho:0, alto:0,
                         r:col.r, g:col.g, b:col.b, a:col.a };
      estadoDibujo = 1;

    } else if (estadoDibujo === 1) {
      // Segundo clic: guarda el cuadro
      if (Math.abs(cuadroTemporal.ancho) > 3 && Math.abs(cuadroTemporal.alto) > 3) {
        cuadros.push(cuadroTemporal);
        document.getElementById('contadorSpan').innerText = 'Cuadros: ' + cuadros.length;
      }
      cuadroTemporal = null;
      estadoDibujo = 0;
    }
    p.redraw();
  };

  // ── Movimiento — actualiza tamaño o detecta hover ──────
  p.mouseMoved = function() {
    if (estadoDibujo === 1 && cuadroTemporal) {
      cuadroTemporal.ancho = p.mouseX - inicioX;
      cuadroTemporal.alto  = p.mouseY - inicioY;
    } else {
      cuadroHover = null;
      for (var i = cuadros.length - 1; i >= 0; i--) {
        var c = cuadros[i];
        var minX = Math.min(c.x, c.x + c.ancho);
        var maxX = Math.max(c.x, c.x + c.ancho);
        var minY = Math.min(c.y, c.y + c.alto);
        var maxY = Math.max(c.y, c.y + c.alto);
        if (p.mouseX >= minX && p.mouseX <= maxX &&
            p.mouseY >= minY && p.mouseY <= maxY) {
          cuadroHover = c;
          break;
        }
      }
      // Info panel
      if (cuadroHover) {
        document.getElementById('info-cuadros').innerText =
          'Hover: Ancho = ' + Math.abs(cuadroHover.ancho) + 'px  ·  Alto = ' +
          Math.abs(cuadroHover.alto) + 'px  (Delete / Supr para eliminar)';
      } else {
        document.getElementById('info-cuadros').innerText = '';
      }
    }
    p.redraw();
  };

  // ── Delete / Supr — eliminar cuadro bajo el cursor ─────
  p.keyPressed = function() {
    if (p.keyCode === p.DELETE && estadoDibujo === 0 && cuadroHover) {
      var idx = cuadros.indexOf(cuadroHover);
      if (idx > -1) {
        cuadros.splice(idx, 1);
        cuadroHover = null;
        document.getElementById('contadorSpan').innerText = 'Cuadros: ' + cuadros.length;
        document.getElementById('info-cuadros').innerText = '';
        p.redraw();
      }
    }
  };
};

new p5(editorSketch);
</script>
""")